In [0]:
# Test cluster is working
print("Cluster is running ")
print(f"Spark version: {spark.version}")
print(f"DBR: {spark.conf.get('spark.databricks.clusterUsageTags.sparkVersion')}")

Cluster is running 
Spark version: 3.4.1
DBR: 13.3.x-scala2.12


In [0]:
# ADLS connection setup
storage_account = "stfleettelemetryalvin"
storage_key = "YOUR_STORAGE_ACCOUNT_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Test connection
dbutils.fs.ls(f"abfss://bronze@{storage_account}.dfs.core.windows.net/")
print("ADLS connection successful")

ADLS connection successful


In [0]:
# Use default Hive metastore (no Unity Catalog needed)
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("CREATE DATABASE IF NOT EXISTS reference")

print("Databases created")
spark.sql("SHOW DATABASES").show()

Databases created
+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|         reference|
|            silver|
+------------------+



In [0]:
# Load vehicle reference dimension
vehicle_data = [
    ("VH001", "James Mitchell",  "R101", "Bangalore-Mysore Express",   "Heavy Truck",    80, 10000, "Bangalore Central"),
    ("VH002", "Robert Carter",   "R102", "Bangalore-Chennai Corridor",  "Medium Van",    100,  3000, "Electronic City"),
    ("VH003", "William Turner",  "R101", "Bangalore-Mysore Express",   "Heavy Truck",    80, 10000, "Bangalore Central"),
    ("VH004", "Thomas Harris",   "R103", "Bangalore-Hyderabad NH44",   "Container Truck", 75, 20000, "Hosur Road Depot"),
    ("VH005", "Daniel Evans",    "R104", "Bangalore-Pune Highway",     "Medium Van",    100,  3000, "Yeshwanthpur"),
    ("VH006", "Christopher Lee", "R102", "Bangalore-Chennai Corridor",  "Light Van",    110,  1500, "Electronic City"),
    ("VH007", "Matthew Wilson",  "R103", "Bangalore-Hyderabad NH44",   "Heavy Truck",    80, 10000, "Hosur Road Depot"),
    ("VH008", "Andrew Thompson", "R105", "Bangalore-Mangalore",        "Container Truck", 75, 20000, "Peenya Industrial"),
    ("VH009", "David Anderson",  "R104", "Bangalore-Pune Highway",     "Light Van",     110,  1500, "Yeshwanthpur"),
    ("VH010", "Richard Moore",   "R105", "Bangalore-Mangalore",        "Medium Van",    100,  3000, "Peenya Industrial"),
]

columns = ["vehicle_id", "driver_name", "route_id", "route_name", 
           "vehicle_type", "max_speed_kmh", "capacity_kg", "home_depot"]

ref_df = spark.createDataFrame(vehicle_data, columns)

ref_df.write.format("delta").mode("overwrite").saveAsTable("reference.vehicle_dimension")

print("Reference table created")
spark.sql("SELECT * FROM reference.vehicle_dimension").show()

Reference table created
+----------+---------------+--------+--------------------+---------------+-------------+-----------+-----------------+
|vehicle_id|    driver_name|route_id|          route_name|   vehicle_type|max_speed_kmh|capacity_kg|       home_depot|
+----------+---------------+--------+--------------------+---------------+-------------+-----------+-----------------+
|     VH007| Matthew Wilson|    R103|Bangalore-Hyderab...|    Heavy Truck|           80|      10000| Hosur Road Depot|
|     VH008|Andrew Thompson|    R105| Bangalore-Mangalore|Container Truck|           75|      20000|Peenya Industrial|
|     VH009| David Anderson|    R104|Bangalore-Pune Hi...|      Light Van|          110|       1500|     Yeshwanthpur|
|     VH010|  Richard Moore|    R105| Bangalore-Mangalore|     Medium Van|          100|       3000|Peenya Industrial|
|     VH003| William Turner|    R101|Bangalore-Mysore ...|    Heavy Truck|           80|      10000|Bangalore Central|
|     VH004|  Thomas Har